In [ ]:
%run main.ipynb

# df_caract_recoder
# df_lieux_recoder
# df_vehicule_recoder
# df_usager_recoder
# df_final

# Modelisation :
- classification avec gravité en variables catégorielle : 1 = Indemne, 2 = Blessé léger, 3 = Blessé hospitalisé, 4 = Tué
- Random Forest 
- Facteurs clés : type de véhicule, ancienneté du véhicule, choc avant / arrière / latéral, âge, sexe, port de la ceinture / casque, alcoolémie, météo, luminosité, état de la chaussée, type de route (autoroute, nationale, départementale), zone (urbaine / rurale), département, zone accidentogène (a rajouter ?), heure, saison (a rajouter ?)
- A faire : one hot encoding, gérer les valeurs manquantes 

In [ ]:
df_final["grav"].unique()

In [ ]:
df_final.columns

## I - Random Forest

In [ ]:
import numpy as np
import random

np.random.seed(66)
random.seed(66)

In [ ]:
grav_dict = {
    "Indemne":1,
    "Tué":4,
    "Blessé hospitalisé":3,
    "Blessé léger":2
}
df_final = recodage(df_final, {"grav": grav_dict})
df_final = df_final.dropna(subset=['grav'])

In [ ]:
df_final.info()

In [ ]:
y = df_final["grav"].astype(int).astype(str)
X = df_final[[
    "mois", "an", "lum", "dep", "agg", "int", "surf", "vma", "catv", "obs",
    "obsm", "choc", "sexe", "situ", "secu1", "age", "atm", "col", "catr",
    "prof", "plan", "manv", "catu", "jour_semaine", "hr"
]]
X["an"] = X["an"].astype(str)

### Vérification des valeurs manquantes - A FAIRE

In [ ]:
X.info()

### Encodage des variables explicatives 

In [ ]:
import pandas as pd

X_encoded = pd.get_dummies(X, drop_first=True)

# Train, test 
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=66, stratify=y
)
# startify pour les classes déséquilibré 


In [ ]:
X_train

### Entrainement du modèle 

A faire : 
- optimisation des paramètres 
- penalisation en fonction des classes (pas bcp de tué)


In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=66,
    n_jobs=-1
)

rf.fit(X_train, y_train)


### Evaluation et importance des facteurs 

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = rf.predict(X_test)

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))




In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# y_test = vraies classes
# y_pred = classes prédites

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
            xticklabels=['Prédit 1', 'Prédit 2','Prédit 3', 'Prédit 4'],
            yticklabels=['Réel 1', 'Réel 2', 'Réel 3', 'Réel 4'])
plt.xlabel('Prédictions')
plt.ylabel('Réel')
plt.title('Matrice de confusion')
plt.show()


In [ ]:
import numpy as np

importances = rf.feature_importances_
indices = np.argsort(importances)[::-1]

for i in indices[:20]:
    print(f"{X_encoded.columns[i]} : {importances[i]:.4f}")

In [ ]:
import numpy as np

cm = confusion_matrix(y_test, y_pred)
cm_norm = cm / np.sum(cm)

plt.figure(figsize=(6,4))
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Purples',
            xticklabels=['Prédit 1', 'Prédit 2','Prédit 3', 'Prédit 4'],
            yticklabels=['Réel 1', 'Réel 2', 'Réel 3', 'Réel 4'])
plt.xlabel('Prédictions')
plt.ylabel('Réel')
plt.title('Matrice de confusion normalisée')
plt.show()


les classes 1 et 2 sont bien prédites, pour les classes 3 et 4 bcp moins debonnes prédictions, cela est du au fait que les données comporte moins d'accidnet comportant ces gravités de blessures, le 3 et 4 sont tué et blessures hospitalisé.
Sauf que nous on voudrait notamment prédire les tués pour voir ce qui est le plus dangereux ! Il va falloir donc rajouter des pénalisation :

In [ ]:
from sklearn.ensemble import RandomForestClassifier

poids = {
    "1": 1,   # indemne
    "2": 1.07,   # blessé léger
    "3": 2.77,   # blessé hospitalisé
    "4": 15.4   # tué
}

rf2 = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight=poids,
    random_state=66,
    n_jobs=-1
)

rf2.fit(X_train, y_train)


In [ ]:
y_pred2 = rf2.predict(X_test)

print(classification_report(y_test, y_pred2))
print(confusion_matrix(y_test, y_pred2))



In [ ]:
from imblearn.ensemble import BalancedRandomForestClassifier

brf = BalancedRandomForestClassifier(
    n_estimators=400,
    sampling_strategy="auto",
    random_state=66,
    n_jobs=-1
)

brf.fit(X_train, y_train)


In [ ]:
y_pred_brf = brf.predict(X_test)

print(classification_report(y_test, y_pred_brf))
print(confusion_matrix(y_test, y_pred_brf))



### GridSearchCV


In [ ]:
param_grid = {
    "n_estimators": [100],   
    "max_depth": [None, 12, 20], 
    "class_weight": [
        {"1":1, "2":1, "3":2, "4":6},
        {"1":1, "2":1, "3":3, "4":15},
        "balanced_subsample"
    ]
}

In [ ]:
from sklearn.model_selection import GridSearchCV

rf_grid = RandomForestClassifier(
    random_state=66,
    n_jobs=1
)

grid = GridSearchCV(
    estimator=rf_grid,
    param_grid=param_grid,
    scoring="recall_macro", # ou f1_weighted
    refit="recall_macro",
    cv=3,
    n_jobs=-1,
    verbose=2
)

grid.fit(X_train, y_train)


In [ ]:
print("Meilleurs paramètres :", grid.best_params_)
print("Meilleurs score :", grid.best_score_)


Selon la gridsearch les meilleurs parametres sont ci-dessus. Nous allons donc refaire un modele avce les paramètres.

In [ ]:
rf_opti = RandomForestClassifier(
    n_estimators=250,
    max_depth=12,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight="balanced_subsample",
    random_state=66,
    n_jobs=-1
)

rf_opti.fit(X_train, y_train)


In [ ]:
y_pred = rf_opti.predict(X_test)

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

on peut égalemtn tester avce plus d'arbres car avec la grid search trop d'arbre prennet trop de ressources donc compliqué par contre nous allons mettre un peu plus d'arbre car a chaque fois, les différentes gridsearch choisisse le plus grand nombre d'arbre disponible 

on remarque que les tué sont bien trouvé en revanche certains non tué sont predit en tué 

In [ ]:
rf_opti2 = RandomForestClassifier(
    n_estimators=400,
    max_depth=12,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight="balanced_subsample",
    random_state=66,
    n_jobs=-1
)

rf_opti2.fit(X_train, y_train)

y_pred = rf_opti2.predict(X_test)

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

In [ ]:
from sklearn.metrics import balanced_accuracy_score
balanced_accuracy_score(y_test, y_pred)


In [ ]:
import pandas as pd
import numpy as np

importances = rf_opti2.feature_importances_
cols = X_train.columns

df_importances = pd.DataFrame({
    "variable": cols,
    "importance": importances
}).sort_values("importance", ascending=False)

df_importances.head(20)


In [ ]:
import shap

# variable influence la gravité
explainer = shap.TreeExplainer(rf_opti2)
shap_values = explainer.shap_values(X_test)



In [ ]:
shap.summary_plot(shap_values, X_test)

In [ ]:
# Trouver l’index de la classe "4" (tué)
classe_tué = list(rf_opti2.classes_).index("4")

# Afficher les SHAP values pour la classe tué
shap.summary_plot(shap_values[classe_tué], X_test)
